In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raesiow/parsed-v2/atkinson_parsed-2.csv


In [2]:
atk_path ='/kaggle/input/datasets/raesiow/parsed-v2/atkinson_parsed-2.csv'
atk = pd.read_csv(atk_path)
print(atk.shape)
atk.head()

(4017, 22)


,food_number,category,subcategory,parse_ok,raw,item,country,year,gi_mean,gi_sem,...,avail_carb_g,test_portion_g,reference_food,study_details_raw,source_table,rep_ref_food,subject_type,n_subjects,healthy_only,page
0,1,BAKERY PRODUCTS,Cakes,True,"1 Cake, NS, decreased GI variant, sugar-to-flo...","Cake, NS, decreased GI variant, sugar-to-flour...",Belgium,2010*,20.0,4.0,...,25,54.4,"Glucose, 2h","Normal, 10 25 54.4 Glucose, 2h Standard Venous...",1,NaN,Normal,10.0,True,2
1,2,BAKERY PRODUCTS,Cakes,True,"2 Carrot cake, prepared with wheat flour and c...","Carrot cake, prepared with wheat flour and coc...",Philippines,2002,37.0,2.0,...,50,NS,"Bread, 2h","Normal, 10 50 NS Bread, 2h Standard Capillary,...",1,NaN,Normal,10.0,True,2
2,3,BAKERY PRODUCTS,Cakes,True,3 Chocolate cake made from packet mix with cho...,Chocolate cake made from packet mix with choco...,USA,2000,38.0,3.0,...,50,92.4,"Glucose, 2h","Normal, 10 50 92.4 Glucose, 2h Standard Capill...",1,NaN,Normal,10.0,True,2
3,4,BAKERY PRODUCTS,Cakes,True,4 Chocolate mudcake Australia 2009 43±4 13 Nor...,Chocolate mudcake,Australia,2009,43.0,4.0,...,50,109.4,"Glucose, 2h","Normal, 9 50 109.4 Glucose, 2h Standard Capill...",1,NaN,Normal,9.0,True,2
4,5,BAKERY PRODUCTS,Cakes,True,5 Christmas fruit cake Australia 2009 53±3 16 ...,Christmas fruit cake,Australia,2009,53.0,3.0,...,50,90.6,"Glucose, 2h","Normal, 10 50 90.6 Glucose, 2h Standard Capill...",1,NaN,Normal,10.0,True,2


In [3]:
# clean name
def clean_name(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.replace("'", "").replace(",", "").replace("(", "").replace(")", "")
    return ' '.join(text.split())

atk['name_clean']= atk['item'].apply(clean_name)

In [4]:
print("empty names:", (atk["name_clean"] == "").sum())
print("double spaces:", atk["name_clean"].str.contains("  ").sum()) 
print("unique names:", atk["name_clean"].nunique(), "of", len(atk))

empty names: 0
double spaces: 0
unique names: 3561 of 4017


In [5]:
import re
def make_core(text):
    if not isinstance(text, str):
        return ''
    text = re.sub('[\u00ae\u2122]',' ', text)
    text = re.sub(r'\s*\([^()]*\)\s*$', '' , text)
    return clean_name(text)

atk['name_core']= atk['item'].apply(make_core)

In [6]:
print("double spaces:", atk["name_clean"].str.contains(r"\s{2,}").sum())
print("double spaces (core):", atk["name_core"].str.contains(r"\s{2,}").sum())

double spaces: 0
double spaces (core): 0


In [7]:
healthy = atk[atk['healthy_only']== True]
print(len(healthy))
print(healthy['source_table'].value_counts())
print(healthy['name_core'].nunique())

3407
source_table
1    2091
2    1316
Name: count, dtype: int64
3061


In [8]:
dup = healthy[healthy.duplicated("name_core", keep=False)]
print("rows in repeated groups:", len(dup))
print("repeated names:", dup["name_core"].nunique())
dup.sort_values("name_core")[["name_core", "gi_mean", "country", "year", "source_table"]].head(20)

rows in repeated groups: 534
repeated names: 188


,name_core,gi_mean,country,year,source_table
2447,all-bran,55.0,USA,2002,2
2444,all-bran,51.0,Canada,1981,2
2443,all-bran,38.0,USA,1981,2
2445,all-bran,51.0,South Korea,2018,2
3059,apple baby food,54.0,Czech Republic,2010,2
3060,apple baby food,54.0,Czech Republic,2005,2
1259,apple baby food,46.0,Czech Republic,2005,1
1086,apricots dried,56.0,China,2018*,1
2898,apricots dried,30.0,Australia,1994,2
1083,apricots raw ns,42.0,Canada,2018*,1


In [9]:
name_spread = healthy.groupby('name_core')['gi_mean'].agg(['count', 'median', 'min', 'max'])
name_spread['range'] = name_spread['max']-name_spread['min']
name_spread = name_spread[name_spread['count']>1]

print(name_spread['range'].describe())
print('name with range > 20:', (name_spread['range'] > 20).sum(), 'of', len(name_spread))
name_spread.sort_values('range', ascending = False).head(10)

count    188.000000
mean       9.962766
std       10.938999
min        0.000000
25%        2.000000
50%        7.000000
75%       15.000000
max       80.000000
Name: range, dtype: float64
name with range > 20: 22 of 188


,count,median,min,max,range
name_core,,,,,
cornflakes,17,79.0,52.0,132.0,80.0
sucrose,15,66.0,59.0,110.0,51.0
yoghurt black cherry,2,42.0,17.0,67.0,50.0
white bread,42,71.0,51.0,94.0,43.0
white rice ns,4,62.0,43.0,83.0,40.0
banana ns,2,52.5,35.0,70.0,35.0
citrus honey,2,63.5,46.0,81.0,35.0
popcorn plain cooked in microwave oven,2,72.0,55.0,89.0,34.0
maltose,3,105.0,75.0,105.0,30.0


In [10]:
pd.set_option("display.max_colwidth", 200)
healthy[healthy["name_core"].str.startswith("barley celebrity")][["food_number", "item", "gi_mean", "gi_sem", "source_table"]]

,food_number,item,gi_mean,gi_sem,source_table
527,528,"Barley, Celebrity cultivar (six-rowed, hulled, normal barley; high amylose, medium ß-glucan), whole grain (only the husk was removed), boiled for 25 min",21.0,4.0,1
528,529,"Barley, Celebrity cultivar (six-rowed, hulled, normal barley; high amylose, medium ß-glucan), pot pearled (all of the bran and most of the germ removed), boiled for 30 min",22.0,3.0,1
532,533,"Barley, Celebrity cultivar (six-rowed, hulled, normal barley; high amylose, medium ß-glucan), commercial pearled (some bran and germ were also removed), boiled for 25 min",25.0,3.0,1
533,534,"Barley, Celebrity cultivar (six-rowed, hulled, normal barley; high amylose, medium ß-glucan) whole grain (only the husk was removed), boiled for 25 min",25.0,4.0,1
540,541,"Barley, Celebrity cultivar (six-rowed, hulled, normal barley; high amylose, medium ß-glucan), white pearled (all of the bran and most of the germ and crease removed), boiled for 30 min",32.0,6.0,1
541,542,"Barley, Celebrity cultivar (six-rowed, hulled, normal barley; high amylose, medium ß-glucan) white pearled (all of the bran and most of the germ and crease removed), boiled for 30 min",33.0,3.0,1


In [11]:
med = healthy.groupby("name_core")["gi_mean"].transform("median")
n = healthy.groupby("name_core")["gi_mean"].transform("count")
dev = (healthy["gi_mean"] - med).abs()[n > 1]

print("mean abs deviation from name median:", dev.mean())
print("median abs deviation:", dev.median())

mean abs deviation from name median: 4.49625468164794
median abs deviation: 2.5


In [12]:
healthy['carb_g']= (pd.to_numeric(healthy['avail_carb_g'], errors='coerce')/pd.to_numeric(healthy['test_portion_g'], errors='coerce')*100)
print('rows with carb per 100g:', healthy['carb_g'].notna().sum(), 'of', len(healthy))

print(healthy['carb_g'].describe())

rows with carb per 100g: 2618 of 3407
count    2618.000000
mean       42.453576
std        25.456498
min         2.702703
25%        18.667182
50%        41.322314
75%        63.291139
max       136.612022
Name: carb_g, dtype: float64


/tmp/ipykernel_16/103051245.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  healthy['carb_g']= (pd.to_numeric(healthy['avail_carb_g'], errors='coerce')/pd.to_numeric(healthy['test_portion_g'], errors='coerce')*100)


In [13]:
bad = healthy["carb_g"] > 100
print("before:", bad.sum(), "rows above 100;", healthy["carb_g"].isna().sum(), "already NaN")

healthy = healthy.copy()
healthy.loc[bad, "carb_g"] = float("nan")

print("after: ", (healthy["carb_g"] > 100).sum(), "rows above 100;", healthy["carb_g"].isna().sum(), "NaN")

before: 5 rows above 100; 789 already NaN
after:  0 rows above 100; 794 NaN


In [14]:
healthy = healthy.copy()
healthy.loc[healthy['carb_g']> 100, 'carb_g'] = np.nan

final = healthy.groupby('name_core').agg(
    gi_median = ('gi_mean', 'median'),
    gi_min = ('gi_mean', 'min'),
    gi_max = ('gi_mean', 'max'),
    n_rows = ('gi_mean', 'count'),
    carb_g = ('carb_g', 'median'),
    example_item = ('item', 'first'),
    any_table1=('source_table', lambda s: (s==1).any()),
).reset_index()

print(len(final), 'rows,', final['name_core'].is_unique)
print(final['gi_median'].isna().sum(), 'missing GI')
final.sort_values('n_rows', ascending=False).head(5)

3061 rows, True
0 missing GI


,name_core,gi_median,gi_min,gi_max,n_rows,carb_g,example_item,any_table1
2830,white bread,71.0,51.0,94.0,42,45.871560,White bread,True
628,cornflakes,79.0,52.0,132.0,17,85.476330,Cornflakes,True
2547,sucrose,66.0,59.0,110.0,15,100.000000,Sucrose,True
2535,strawberry fruit bar kettle valley,33.0,30.0,36.0,8,72.463768,"Strawberry Fruit Bar, Kettle Valley (SunOpta, Inc., Brampton, ON, Canada)",True
1567,montana chocolate chip cookies,42.0,41.0,43.0,8,79.113924,"Montana Chocolate Chip Cookies (Kinnikinnic Foods Inc., Edmonton. AB, Canada)",True


In [15]:
final.info()
final.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3061 entries, 0 to 3060
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name_core     3061 non-null   object 
 1   gi_median     3061 non-null   float64
 2   gi_min        3061 non-null   float64
 3   gi_max        3061 non-null   float64
 4   n_rows        3061 non-null   int64  
 5   carb_g        2388 non-null   float64
 6   example_item  3061 non-null   object 
 7   any_table1    3061 non-null   bool   
dtypes: bool(1), float64(4), int64(1), object(2)
memory usage: 170.5+ KB


,name_core,gi_median,gi_min,gi_max,n_rows,carb_g,example_item,any_table1
0,"""mai-lai"" cake instant prepared freshly in the morning",61.0,61.0,61.0,1,43.859649,"""Mai-Lai"" Cake, Instant, prepared freshly in the morning",True
1,"""pineapple"" bun instant",65.0,65.0,65.0,1,54.945055,"""Pineapple"" bun, instant (Hoixe Cake Shop, H.K.)",True
2,100 g glucose + 20 g acacia gum,85.0,85.0,85.0,1,100.000000,100 g Glucose + 20 g acacia gum,False
3,100 g glucose + 20 g cellulose,103.0,103.0,103.0,1,100.000000,"100 g glucose + 20 g cellulose (CSIR Biochemical Unit, New Delhi, India)",False
4,100% durum semolina spaghetti boiled 15 min,41.0,41.0,41.0,1,69.156293,"100% durum semolina spaghetti, boiled 15 min (Unico, Canada)",False


In [16]:
final.to_csv('/kaggle/working/atkinson_main_unique.csv', index=False)
